# 🔄 Basic Agent Workflows with Azure OpenAI (Python)

## 📋 Workflow Orchestration Tutorial

This notebook introduces the powerful **Workflow Builder** capabilities of the Microsoft Agent Framework. Learn how to create sophisticated, multi-step agent workflows that can handle complex business processes and coordinate multiple AI operations seamlessly.

## 🎯 Learning Objectives

### 🏗️ **Workflow Architecture**
- **Workflow Builder**: Design and orchestrate complex multi-step processes
- **Event-Driven Execution**: Handle workflow events and state transitions
- **Visual Workflow Design**: Create and visualize workflow structures
- **Azure OpenAI Integration**: Leverage AI models within workflow contexts

### 🔄 **Process Orchestration**
- **Sequential Operations**: Chain multiple agent tasks in logical order
- **Conditional Logic**: Implement decision points and branching workflows
- **Error Handling**: Robust error recovery and workflow resilience
- **State Management**: Track and manage workflow execution state

### 📊 **Enterprise Workflow Patterns**
- **Business Process Automation**: Automate complex organizational workflows
- **Multi-Agent Coordination**: Coordinate multiple specialized agents
- **Scalable Execution**: Design workflows for enterprise-scale operations
- **Monitoring & Observability**: Track workflow performance and outcomes

## ⚙️ Prerequisites & Setup

### 📦 **Required Dependencies**

Install the Agent Framework with workflow capabilities:

```bash
pip install agent-framework-core -U
```

### 🔑 **Azure OpenAI Configuration**

**Environment Setup (.env file):**
```env
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-4o-mini
AZURE_OPENAI_API_KEY=your_azure_openai_api_key
AZURE_OPENAI_API_VERSION=2024-12-01-preview
```

### 🏢 **Enterprise Use Cases**

**Business Process Examples:**
- **Customer Onboarding**: Multi-step verification and setup workflows
- **Content Pipeline**: Automated content creation, review, and publishing
- **Data Processing**: ETL workflows with AI-powered transformation
- **Quality Assurance**: Automated testing and validation processes

**Workflow Benefits:**
- 🎯 **Reliability**: Deterministic execution with error recovery
- 📈 **Scalability**: Handle high-volume process automation
- 🔍 **Observability**: Complete audit trails and monitoring
- 🔧 **Maintainability**: Visual design and modular components

## 🎨 Workflow Design Patterns

### Basic Workflow Structure
```mermaid
graph TD
    A[Start] --> B[Agent Task 1]
    B --> C{Decision Point}
    C -->|Success| D[Agent Task 2]
    C -->|Failure| E[Error Handler]
    D --> F[End]
    E --> F
```

**Key Components:**
- **WorkflowBuilder**: Main orchestration engine
- **WorkflowEvent**: Event handling and communication
- **WorkflowViz**: Visual workflow representation and debugging

Let's build your first intelligent workflow! 🚀

In [ ]:
# ✅ Dependencies installed via terminal:
# uv venv .venv --python 3.13
# uv pip install ipykernel --python .venv/Scripts/python.exe
# uv pip install -r Installation/requirements.txt --python .venv/Scripts/python.exe
#
# Select the .venv kernel in VS Code to use these packages.

In [15]:
# 🔄 Import Workflow and Agent Framework Components
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework import AgentResponse, WorkflowBuilder, WorkflowEvent, WorkflowViz

In [16]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from typing import cast
from dotenv import load_dotenv # 📁 Secure configuration loading

In [17]:
# 🔧 Initialize Environment Configuration
from dotenv import find_dotenv
load_dotenv(find_dotenv())

True

In [18]:
# 🔗 Create Azure OpenAI Chat Client for Workflow Operations
chat_client = OpenAIChatCompletionClient(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    model=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)

In [19]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [20]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [21]:
reviewer_agent   = chat_client.as_agent(
        instructions=(
           REVIEWER_INSTRUCTIONS
        ),
        name=REVIEWER_NAME,
    )

front_desk_agent = chat_client.as_agent(
        instructions=(
            FRONTDESK_INSTRUCTIONS
        ),
        name=FRONTDESK_NAME,
    )

In [22]:
workflow = WorkflowBuilder(start_executor=front_desk_agent).add_edge(front_desk_agent, reviewer_agent).build()

In [23]:
# Workflow visualization (Mermaid output only — SVG export requires Graphviz system binary, not needed to run the workflow)
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")

# SVG export — not needed, requires graphviz system binary (winget install --id Graphviz.Graphviz)
# svg_file = viz.export(format="svg")
# print(f"SVG file saved to: {svg_file}")

Generating workflow visualization...
Mermaid string: 
flowchart TD
  FrontDesk["FrontDesk (Start)"];
  Concierge["Concierge"];
  FrontDesk --> Concierge;


In [24]:
class DatabaseEvent(WorkflowEvent): ...

In [25]:
# SVG display — not needed, requires graphviz system binary
# from IPython.display import SVG, display, HTML
# if svg_file and os.path.exists(svg_file):
#     display(SVG(filename=svg_file))

In [26]:
events = await workflow.run("I would like to go to Paris.")

outputs = events.get_outputs()
    # The outputs of the workflow are whatever the agents produce. So the outputs are expected to be a list
    # of `AgentResponse` from the agents in the workflow.
outputs = cast(list[AgentResponse], outputs)
for output in outputs:
    print(f"{output.messages[0].author_name}: {output.text}\n")

    # Summarize the final run state (e.g., COMPLETED)
print("Final state:", events.get_final_state())

FrontDesk: Eiffel Tower summit at sunset—prebook timed tickets on the official site and arrive 45–60 minutes before sunset to catch daylight, golden hour, and the first hourly sparkle.

Concierge: Not approved.

How to refine the recommendation:
- Prioritize resident-centric experiences over iconic landmarks at peak photo hours; aim for activities where locals clearly outnumber visitors.
- Match the plan to your interests (food, music, crafts, sport, design) and choose participatory formats led by residents rather than passive sightseeing.
- Shift timing to off-peak windows (weekday mornings or late evenings outside golden hour) to avoid queues and crowd clustering.
- Explore beyond the most central districts, favoring walkable, residential areas with independent venues and everyday street life.
- Keep group sizes small or go self-guided; verify authenticity by favoring offerings presented primarily in the local language and booked directly.
- Leave unstructured time to linger, observe